# QAOA depth sweep

Sweep `p in {1..5}` for a fixed problem and report approximation ratio,
feasible-state probability, and probability of the brute-force optimum.
This is the headline plot for the QAOA section.

### SETUP

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np

from scripts.data      import load_returns
from scripts.baskets   import config
from scripts.portfolio import Portfolio
from scripts.qaoa      import QAOA
from scripts.classical import brute_force
from scripts.metrics   import approximation_ratio, prob_optimal, prob_feasible

RESULTS_DIR = Path.cwd().parent / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

### PROBLEM

In [ ]:
BASKET = 'mag7'
K, LAM, AP = 2, 2.0, 0.5
P_VALUES   = [1, 2, 3, 4, 5]
N_RESTARTS = 15

tickers, start, end = config(BASKET)
r  = load_returns(tickers, start, end, cache_name=BASKET)
pf = Portfolio(r.mu, r.Sigma, lam=LAM, A=AP, K=K, tickers=list(r.tickers))

bf   = brute_force(pf)
qaoa = QAOA(pf, seed=42)
E0   = qaoa.ground_state_energy()
print(f'brute force: x={bf.bitstring}  C={bf.cost:.6f}  E0={E0:.6f}')

### SWEEP (cached to `results/depth_sweep_<basket>.json`)

In [ ]:
cache = RESULTS_DIR / f'depth_sweep_{BASKET}.json'

if cache.exists():
    sweep = json.loads(cache.read_text())
    print(f'loaded cached sweep from {cache.name}')
else:
    sweep = []
    for p in P_VALUES:
        res = qaoa.optimise(p=p, n_restarts=N_RESTARTS)
        sweep.append({
            'p':            p,
            'energy':       res['energy'],
            'ratio':        approximation_ratio(res['energy'], E0),
            'p_optimal':    prob_optimal(res['probs'], bf.x),
            'p_feasible':   prob_feasible(res['probs'], pf.n, pf.K),
        })
    cache.write_text(json.dumps(sweep, indent=2))
    print(f'saved → {cache.name}')

import pandas as pd
pd.DataFrame(sweep)